In [2]:
"""
statsbomb_explore.py

Safe explorer + loader for the StatsBomb open-data repo structure:

    open-data/data/
        competitions.json
        matches/{competition_id}/{season_id}.json
        events/{match_id}.json
        lineups/{match_id}.json
        three-sixty/{match_id}.json

Usage:
    conda activate your_env
    pip install pandas

    python statsbomb_explore.py "F:\\messi_heatmap\\open-data\\data"
"""

import sys
import os
import json
import pandas as pd

# --- SET YOUR PATH HERE ---
# If running from a normal terminal, you can still pass it as: python statsbomb_explore.py "path"
# If running inside Jupyter, just edit the default string below directly.
DEFAULT_DATA_DIR = r"F:\messi_heatmap\open-data\data"

def _get_data_dir():
    # Only use sys.argv if it looks like a real path (not Jupyter's "-f ...")
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
        return sys.argv[1]
    return DEFAULT_DATA_DIR

DATA_DIR = _get_data_dir()


def load_competitions():
    path = os.path.join(DATA_DIR, "competitions.json")
    with open(path, encoding="utf-8") as f:
        comps = json.load(f)
    df = pd.DataFrame(comps)
    return df


def load_matches(competition_id, season_id):
    """Load match metadata for one competition/season (small file)."""
    path = os.path.join(DATA_DIR, "matches", str(competition_id), f"{season_id}.json")
    with open(path, encoding="utf-8") as f:
        matches = json.load(f)
    # flatten nested dicts like home_team, away_team, competition, season
    df = pd.json_normalize(matches, sep="_")
    return df


def load_events(match_id):
    """Load the full event stream for ONE match. Safe: single match files are small."""
    path = os.path.join(DATA_DIR, "events", f"{match_id}.json")
    with open(path, encoding="utf-8") as f:
        events = json.load(f)
    df = pd.json_normalize(events, sep="_")
    return df


def load_lineups(match_id):
    path = os.path.join(DATA_DIR, "lineups", f"{match_id}.json")
    with open(path, encoding="utf-8") as f:
        lineups = json.load(f)
    return lineups  # list of 2 team dicts with 'lineup' (list of players)


def iter_events_for_season(competition_id, season_id, columns=None):
    """
    Generator: yields one DataFrame of events per match in a competition/season,
    one match at a time, so memory stays low even across hundreds of matches.

    Example:
        for match_id, events_df in iter_events_for_season(11, 90):
            # process one match's events here
            ...
    """
    matches_df = load_matches(competition_id, season_id)
    for match_id in matches_df["match_id"]:
        try:
            ev = load_events(match_id)
            if columns:
                ev = ev[[c for c in columns if c in ev.columns]]
            yield match_id, ev
        except FileNotFoundError:
            continue


def main():
    print(f"Data dir: {DATA_DIR}\n")

    comps = load_competitions()
    print(f"Competitions available: {len(comps)}")
    print(comps[["competition_id", "season_id", "competition_name", "season_name"]].to_string(index=False))

    # quick file size sanity check
    events_dir = os.path.join(DATA_DIR, "events")
    if os.path.isdir(events_dir):
        n_files = len(os.listdir(events_dir))
        sample_files = os.listdir(events_dir)[:5]
        print(f"\n'events' folder contains {n_files} match files.")
        for f in sample_files:
            fp = os.path.join(events_dir, f)
            print(f"  {f}: {os.path.getsize(fp)/1e6:.2f} MB")

        # preview ONE match's events in detail
        first_match_id = os.path.splitext(sample_files[0])[0]
        print(f"\n--- Preview of one match's events (match_id={first_match_id}) ---")
        ev = load_events(first_match_id)
        print(f"Shape: {ev.shape}")
        print(f"Columns ({len(ev.columns)}): {list(ev.columns)[:25]} ...")
        print(ev[["type_name", "player_name", "minute"]].head(10)
              if "type_name" in ev.columns else ev.head())


if __name__ == "__main__":
    main()

Data dir: F:\messi_heatmap\open-data\data

Competitions available: 80
 competition_id  season_id        competition_name season_name
              9        281           1. Bundesliga   2023/2024
              9         27           1. Bundesliga   2015/2016
           1267        107  African Cup of Nations        2023
             16          4        Champions League   2018/2019
             16          1        Champions League   2017/2018
             16          2        Champions League   2016/2017
             16         27        Champions League   2015/2016
             16         26        Champions League   2014/2015
             16         25        Champions League   2013/2014
             16         24        Champions League   2012/2013
             16         23        Champions League   2011/2012
             16         22        Champions League   2010/2011
             16         21        Champions League   2009/2010
             16         41        Champions Leag

In [3]:
"""messi_extract_full_career.py

Builds a full-career Messi location dataset from your LOCAL StatsBomb
open-data folder, covering every competition present locally
(not just La Liga) — Champions League, Copa del Rey, Copa America,
World Cup, etc. Each row is tagged with competition/team so you can
filter club-only vs international-only vs everything later.

Much faster than the old version because everything is read from
local disk instead of fetched over the network.

Usage (Jupyter):
    import messi_extract_full_career as mx
    df = mx.run()
"""

import os
import json
import pandas as pd

DATA_DIR = r"F:\messi_heatmap\open-data\data"

# Teams to track. Barcelona = club career, Argentina = international career.
TARGET_TEAMS = {"Barcelona", "Argentina"}
PLAYER_MATCH = "Messi"


def load_competitions():
    path = os.path.join(DATA_DIR, "competitions.json")
    with open(path, encoding="utf-8") as f:
        comps = json.load(f)
    return pd.DataFrame(comps)


def find_relevant_matches():
    """
    Scan EVERY competition/season available locally, and collect matches
    involving Barcelona or Argentina. Only reads small 'matches' files,
    not events, so this step is fast even across 80 competition/seasons.
    """
    comps = load_competitions()
    relevant = []

    for _, row in comps.iterrows():
        comp_id = row["competition_id"]
        season_id = row["season_id"]
        path = os.path.join(DATA_DIR, "matches", str(comp_id), f"{season_id}.json")
        if not os.path.exists(path):
            continue  # not downloaded locally, skip cleanly

        with open(path, encoding="utf-8") as f:
            matches = json.load(f)

        for m in matches:
            home = m["home_team"]["home_team_name"]
            away = m["away_team"]["away_team_name"]
            team_hit = None
            if home in TARGET_TEAMS:
                team_hit = home
            elif away in TARGET_TEAMS:
                team_hit = away
            if team_hit is None:
                continue

            relevant.append({
                "match_id": m["match_id"],
                "match_date": m.get("match_date"),
                "competition_id": comp_id,
                "season_id": season_id,
                "competition_name": row["competition_name"],
                "season_name": row["season_name"],
                "team": team_hit,
                "competition_type": "club" if team_hit == "Barcelona" else "international",
            })

    return pd.DataFrame(relevant)


def extract_messi_events(match_row):
    """Load one match's events locally and pull out Messi's located events."""
    match_id = match_row["match_id"]
    path = os.path.join(DATA_DIR, "events", f"{match_id}.json")
    if not os.path.exists(path):
        return []

    with open(path, encoding="utf-8") as f:
        events = json.load(f)

    rows = []
    for e in events:
        player = e.get("player", {})
        if not player or PLAYER_MATCH not in player.get("name", ""):
            continue
        loc = e.get("location")
        if not loc:
            continue
        rows.append({
            "match_id": match_id,
            "match_date": match_row["match_date"],
            "competition_name": match_row["competition_name"],
            "season_name": match_row["season_name"],
            "competition_type": match_row["competition_type"],
            "team": match_row["team"],
            "x": loc[0],
            "y": loc[1],
            "event_type": e.get("type", {}).get("name"),
            "minute": e.get("minute"),
        })
    return rows


def run(save_dir="../data"):
    print("Step 1: scanning all competitions/seasons for Barcelona/Argentina matches...")
    matches_df = find_relevant_matches()
    print(f"Found {len(matches_df)} relevant matches across "
          f"{matches_df['competition_name'].nunique()} competitions.")
    print(matches_df.groupby(["competition_name", "season_name"]).size())

    print("\nStep 2: extracting Messi's events from each match locally...")
    all_rows = []
    missing = 0
    for i, match_row in matches_df.iterrows():
        rows = extract_messi_events(match_row)
        if not rows:
            missing += 1
        all_rows.extend(rows)
        if (i + 1) % 200 == 0:
            print(f"  {i+1}/{len(matches_df)} matches processed...")

    print(f"\nTotal Messi located events: {len(all_rows)}")
    print(f"Matches with no Messi events found (missing file, or he didn't play): {missing}")

    df = pd.DataFrame(all_rows)
    df["x"] = pd.to_numeric(df["x"])
    df["y"] = pd.to_numeric(df["y"])
    df["match_date"] = pd.to_datetime(df["match_date"])
    df = df.sort_values(["match_date", "minute"])

    os.makedirs(save_dir, exist_ok=True)
    csv_path = os.path.join(save_dir, "messi_locations_full_career.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")

    return df


if __name__ == "__main__":
    run()

Step 1: scanning all competitions/seasons for Barcelona/Argentina matches...
Found 553 relevant matches across 5 competitions.
competition_name  season_name
Champions League  2008/2009       1
                  2010/2011       1
                  2014/2015       1
Copa America      2024            6
Copa del Rey      1977/1978       1
                  1982/1983       1
                  1983/1984       1
FIFA World Cup    1974            1
                  1986            3
                  1990            1
                  2018            4
                  2022            7
La Liga           1973/1974       1
                  2004/2005       7
                  2005/2006      17
                  2006/2007      26
                  2007/2008      27
                  2008/2009      31
                  2009/2010      35
                  2010/2011      33
                  2011/2012      37
                  2012/2013      32
                  2013/2014      31
               